# Covalently modify a protein, watch it relax, and simulate it
### Joseph R. Laforet Jr.

This is the tutorial for `attach`: use this when you want to put a fragment you
wrote as SMILES onto a protein and hand the result to OpenFF.
1. **Load a protein** with full chemistry (notebook 1 wrote the input).
2. **Build a fragment from scratch** as star-sited SMILES. The `*` marks
   where the bond forms.
3. **Deprotonate the attachment site.** The reactive form of a lysine side
   chain is the neutral amine, not the ammonium ion.
4. **Attach them.** One call places the fragment, removes one hydrogen per
   side, and forms the bond.
5. **Look at the result**, then **relax the fragment as a movie**. The
   protein stays fixed and only the fragment moves.
6. **Export** the modified PDB plus one bond record per new bond.
7. **Parameterize.** Those two outputs are all openff-pablo needs, and from
   there Interchange and OpenMM take over.

Run notebook 1 first, so `1ubq_protonated.pdb` exists. Notebooks 3 and 4
build on this one: point mutations, and reactions that do more than replace
one bond.

In [ ]:
from mbuild.biopolymers import Protein, prepare_fragment

import demo_utils

mbuild_protein = Protein("1ubq_protonated.pdb")
print(len(list(mbuild_protein.residues())), "residues | net formal charge:",
      mbuild_protein.net_formal_charge)

## 1. Prepare the fragment

Write the fragment as SMILES with a `*` at the atom that forms the bond.
`prepare_fragment` turns the star into a hydrogen, keeps the charges written
in the SMILES, gives every atom a PDB-style name, and records the bond site
in `link_atoms`. Here: an octanoyl group, starred at the carbonyl carbon.

Choose the residue code with care. A fragment code must not match a code
that the wwPDB Chemical Component Dictionary already assigns. This demo
checked `OC8` against the RCSB and found it unassigned. A code that
collides, `OCT` for n-octane for example, lets a downloaded CCD template
take the place of the fragment definition, and the modified structure then
fails to read back.

In [ ]:
mbuild_fragment = prepare_fragment("*C(=O)CCCCCCC", "OC8")
print("bond site:", mbuild_fragment.link_atoms)
print(mbuild_fragment.n_particles, "atoms |",
      [particle.name for particle in mbuild_fragment.particles()][:6], "...")

## 2. Deprotonate the attachment site

At pH 7 a lysine side chain carries a protonated ammonium group. That group
has no lone pair, so it does not attack the acyl carbon and it is not the
reactive form of the side chain. A conjugation reaction is run under
conditions that favour the neutral free amine, and the reaction replaces one
N-H bond of that amine with the new bond.

`Protein.deprotonate` removes one proton from the named atom and re-matches
the residue to the Chemical Component Dictionary variant that describes the
result. The template, the residue formal charge and the per-atom formal
charges then all agree with the structure.

In [ ]:
charge_before = mbuild_protein.net_formal_charge
mbuild_protein.deprotonate(63, "NZ", chain_id="A")

mbuild_lysine = mbuild_protein.get_residue(63, chain_id="A")
mbuild_nz_atom = mbuild_protein.get_atom(63, "NZ", chain_id="A")
print("LYS 63 formal charge:", mbuild_lysine.formal_charge,
      "| bonded to NZ:", sorted(atom.name for atom in mbuild_nz_atom.direct_bonds()))
print("net formal charge:", charge_before, "->", mbuild_protein.net_formal_charge)

## 3. Attach it

Name the protein site by residue number and atom name. The fragment already
knows its own bond site from the `*`. One hydrogen leaves each side, ports
along the two removed-hydrogen vectors align the fragment, and the bond
forms.

The net formal charge stays at -1 across the new bond. The bond takes the
place of an N-H bond, so the nitrogen keeps its charge of zero and the
product is a neutral secondary amide. The charge fell from 0 to -1 at the
deprotonation, when the site lost its proton, so each modified lysine lowers
the net charge of the protein by one.

`attach` relaxes the fragment when it lands too close to the protein. Here
`relax=False` turns that off, because the next steps show the relaxation as
a movie. The warning about close atoms is the expected result of that
choice.

In [ ]:
bond_record = mbuild_protein.attach(mbuild_fragment, resnum=63, atom_name="NZ",
                        chain_id="A", relax=True)
print("new bond:", bond_record.residue1.name, bond_record.atom1_name, "-",
      bond_record.residue2.name, bond_record.atom2_name)
print("leaving hydrogens:", bond_record.leaving1, bond_record.leaving2)
print("LYS 63 formal charge:",
      mbuild_protein.get_residue(63, chain_id="A").formal_charge,
      "| net formal charge:", mbuild_protein.net_formal_charge)

## 4. Look at the modification

`Protein.save_pdb` writes the file that every downstream loader reads, so the
view is built from that text. `show_protein` takes the linkage residues from
`bond_records()` and the fragment residues from the HETATM flag, so it needs
no further argument.

The protein is a grey cartoon, the modified LYS 63 is cyan sticks, the new
OC8 residue is green sticks, and the view centers on the fragment.

In [ ]:
demo_utils.show_protein(mbuild_protein)

## 5. Relax the fragment as a movie

`protein.relax_fragments()` is the one-call form: it minimizes every HETATM
residue with mBuild's generic parameters while every other atom carries zero
mass, so the protein coordinates do not change.

`demo_utils.relax_movie` runs that same minimization in short bursts and
saves the coordinates after each burst, so the frames become a movie. Frame
0 is the rigid placement, before any minimization. Most of the motion
happens in the first frames, so the movie shows the fragment swing out of
the clash and settle.

The colors mean the same as in the picture above. Use the always-visible
play/pause buttons and Frame slider below the canvas. `relax_movie` returns
unpackable movie data that also displays a player when it is the last
expression in a cell; `show_movie` can display the saved file or that data.

In [ ]:
import numpy as np

demo_movie = demo_utils.relax_movie(mbuild_protein, "octanoyl_relax.pdb",
                               n_frames=40, steps_per_frame=5)
path, frames, energies = demo_movie
step = np.linalg.norm(frames[1:] - frames[:-1], axis=2).max(axis=1)
print(path, "|", len(frames), "frames |", frames.shape[1], "atoms")
print("largest atom step per frame (A), frames 1-3:", np.round(step[:3], 2))
print("potential energy (kJ/mol): first", round(float(energies[0])),
      "-> last", round(float(energies[-1])))

nglview_widget = demo_utils.show_movie(demo_movie, protein=mbuild_protein)
nglview_widget

## 6. Write the modified PDB and the bond records

`save_pdb` writes a standards-conformant file: residue names, real PDB
residue numbers, chain identifiers, HETATM for the fragment, TER after each
chain, and CONECT records only for the bonds that residue adjacency cannot
imply.

`bond_records()` returns one plain dict per new bond. It names the two
residues, the two bonded atoms, the hydrogens that left each side, and the
bond order. That is everything a downstream loader needs to know about the
modification.

In [ ]:
mbuild_protein.save_pdb("1ubq_octanoyl.pdb", overwrite=True)
bond_records = mbuild_protein.bond_records()
bond_records

## 7. Ingest with OpenFF Pablo

Pablo reads the PDB file against residue definitions. It knows the CCD
residues, so it needs two things from us: a definition for `OC8`, and a
declaration of the crosslink.

The definition comes from the same starred SMILES that built the fragment,
with the star turned back into the hydrogen that left. The crosslink comes
straight from the bond record: the two residues, the two atoms, the atoms
absent from the file because the bond exists, and the bond order. No
chemistry is added at this step; the record already holds it.

In [ ]:
from openff.pablo import STD_CCD_CACHE, ResidueDefinition, topology_from_pdb
from openff.toolkit import Molecule
from rdkit import Chem

# Build the OC8 definition from the SAME starred SMILES, with the star
# replaced by hydrogen, exactly as prepare_fragment did. The atom order
# then matches, so the names transfer by position.
rdkit_star_mol = Chem.RWMol(Chem.MolFromSmiles("*C(=O)CCCCCCC"))
for atom in rdkit_star_mol.GetAtoms():
    if atom.GetAtomicNum() == 0:
        atom.SetAtomicNum(1)
rdkit_mol = rdkit_star_mol.GetMol()
Chem.SanitizeMol(rdkit_mol)
openff_fragment_molecule = Molecule.from_rdkit(Chem.AddHs(rdkit_mol), allow_undefined_stereo=True)

# Zip against the pristine `mbuild_fragment`, not against the OC8 residue
# inside the protein: attach() cloned the fragment and removed H1 from the
# clone, so the residue holds one atom less and every name after H1 would shift.
for atom, particle in zip(openff_fragment_molecule.atoms, mbuild_fragment.particles()):
    atom.name = particle.name
oc8_definition = ResidueDefinition.from_molecule(openff_fragment_molecule, residue_name="OC8")

# The crosslink, read off the bond record.
record = bond_records[0]
pablo_residue_library = STD_CCD_CACHE.with_({"OC8": [oc8_definition]}).with_crosslink(
    residues=record["residue_names"],            # ("LYS", "OC8")
    linking_atoms=record["atom_names"],          # ("NZ", "C1")
    leaving_atoms=record["leaving_atoms"],       # (["HZ1"], ["H1"]): absent because the bond exists
    bond_order=record["bond_order"],
)

openff_topology = topology_from_pdb("1ubq_octanoyl.pdb", residue_library=pablo_residue_library)
openff_molecule = openff_topology.molecule(0)
print(openff_molecule.n_atoms, "atoms | net charge:", openff_molecule.total_charge)

## 8. Assign force-field parameters

We use the OpenFF ecosystem to assign parameters. 

**(NOTE: When OpenFF Rosemary is ready, this step becomes unnecessary)**

`demo_charges.assign_split_charges` gives every atom of the conjugate a
partial charge from one of two models. Every atom of a standard residue
keeps its Amber **ff14SB** library charge, read from the unmodified protein.
The fragment and the modified lysine take **NAGL am1bcc graph charges**
(`openff-gnn-am1bcc-0.1.0-rc.3`), computed on a capped model of the
modification site. For this, we take the modified residue and the fragment, every atom within two bonds of them, with a hydrogen closing each bond that was cut, and then assign NAGL charges. The seam between the two models therefore lies on the
peptide bonds of the modified residue. The two sets do not sum to the formal
charge exactly, so the function spreads the small residual over the atoms of
that scope and prints it.

The table reports, per atom of the site, the ff14SB charge, the graph charge,
the final charge, and `delta = NAGL - ff14SB` **before** the residual
correction. `final = NAGL + residual / number_of_site_atoms`. A dash means
there is no corresponding atom in the unmodified reference. `OC877` means
residue name OC8, residue number 77. All values are in elementary charge.

In [ ]:
import demo_charges

# The protein before the modification supplies the ff14SB library
# charges. It is the file that notebook 1 wrote.
openff_unmodified_topology = topology_from_pdb("1ubq_protonated.pdb")

openff_conjugate_molecule = demo_charges.assign_split_charges(
    openff_molecule, openff_unmodified_topology, fragment_resname="OC8",
    resnum=63, chain_id="A")

## 9. Solvate and run MD with OpenMM

`demo_charges.parameterize_with_preset_charges` is a helper function that assigns **ff14SB** to every
atom within the unmodified protein and **Sage 2.3.0** to every
atom involving the modified lysine or fragment. This includes bonds,
angles, and torsions across the site boundary. Parameters outside the site
are transferred from the unmodified ff14SB reference, so the split does not
depend on force-field file order. Lennard–Jones parameters follow the same
atom selection. Solvent uses Sage’s TIP3P parameters. Constraints on bonds to hydrogen
 use the selected force field’s equilibrium bond lengths.

It builds the Interchange with
the split charges passed in as preset charges. It then reads the charges back
out of the Interchange and compares them to the array it passed in. That
check is needed because Sage 2.3.0 carries its own NAGLCharges handler, which
would otherwise compute new charges for the whole conjugate.

**(NOTE: When OpenFF Rosemary is ready, this step becomes unnecessary, or rather much simpler)**

In [ ]:
from openff.interchange.components._packmol import UNIT_CUBE, pack_box
from openff.toolkit import Topology
from openff.units import unit as off_unit

openff_water = Molecule.from_smiles("O")
openff_water.generate_conformers(n_conformers=1)
for atom in openff_water.atoms:
    atom.metadata["residue_name"] = "HOH"

openff_solvated_topology = pack_box([openff_water], [1500],
                    solute=Topology.from_molecules([openff_conjugate_molecule]),
                    target_density=0.95 * off_unit.gram / off_unit.milliliter,
                    box_shape=UNIT_CUBE,
                    tolerance=2.0 * off_unit.angstrom)
print("solvated:", openff_solvated_topology.n_atoms, "atoms")

openff_interchange = demo_charges.parameterize_with_preset_charges(
    openff_solvated_topology, openff_conjugate_molecule, openff_unmodified_topology,
    fragment_resname="OC8", resnum=63, chain_id="A")
print("parameterized:", openff_interchange.topology.n_atoms, "atoms")

In [ ]:
import openmm
from openmm import unit
from datetime import datetime
from tempfile import NamedTemporaryFile
import mdtraj
import nglview

openmm_simulation = openff_interchange.to_openmm_simulation(
    integrator=openmm.LangevinMiddleIntegrator(
        300 * unit.kelvin, 
        1.0 / unit.picosecond, 
        2.0 * unit.femtosecond),
    platform = openmm.Platform.getPlatformByName('CUDA'))
openmm_simulation.minimizeEnergy(maxIterations=200)
openmm_simulation.context.setVelocitiesToTemperature(300 * unit.kelvin)


steps =               100_000 # recall 2fs time step
report_interval =       1_000


openmm_simulation.reporters.append(
    openmm.app.DCDReporter(file="trajectory.dcd", reportInterval=report_interval)
)
print(f"{datetime.now()} Simulating...")
openmm_simulation.step(steps)


In [ ]:
print(f"{datetime.now()} Visualizing.")
trajectory: mdtraj.Trajectory = mdtraj.load(
    "trajectory.dcd", top=mdtraj.Topology.from_openmm(openff_interchange.to_openmm_topology())
)

view = nglview.show_mdtraj(trajectory)
view.add_representation("licorice", selection="63:A or [OC8]")
view.add_representation("unitcell")
view

### It should wiggle! NGLView was giving me problems for a bit, so let me know if it's broken.
If NGLView doesn't cooperate, you can write the Interchange to PDB file format, load that into PyMol (or whatever you use), and then load in the `trajectory.dcd` file we generated.

In [ ]:
openff_interchange.to_pdb("02_demo_topology.pdb")

## Recap

- **mBuild** is responsible for the coordinates and topology: strict protein loading, fragment
  definition from SMILES, deprotonation of the attachment site, covalent
  attachment, fragment relaxation with the protein fixed, and
  chemistry-complete export.
- **OpenFF** is responsible for the parameters: Pablo ingestion from the PDB file plus
  one bond record, ff14SB library charges on the standard residues with NAGL
  am1bcc graph charges on the modified site. Unmodified protein interaction
  terms use ff14SB; terms touching the modified residue or fragment use Sage
  2.3.0. Interchange exports the assembled system to OpenMM.